# Model: Random Forest

Owner: **Daniel**

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

INCLUDE_RADIATION_DAY_BEFORE = True  # HEY TEAM I ADDED THIS LIUNE HERE

MODEL_NAME = "random_forest" if INCLUDE_RADIATION_DAY_BEFORE else "random_forest_no_radiation_day_before"

In [2]:
# using the shared train/validation/test sets

DATA_DIR = Path("../../data/NSW")
RESULTS_DIR = Path("../../results")
RESULTS_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "nsw_train.csv", parse_dates=["DATETIME"])
validation = pd.read_csv(DATA_DIR / "nsw_validation.csv", parse_dates=["DATETIME"])
test = pd.read_csv(DATA_DIR / "nsw_test.csv", parse_dates=["DATETIME"])

print(train.shape, validation.shape, test.shape)

(140208, 55) (17520, 55) (17520, 55)


In [ ]:
# dropping columns

TARGET = "TOTALDEMAND"
DROP_COLS = ["DATETIME", TARGET, "TEMPERATURE", "radiation", "forecast_closest", "forecast_12hr_prior", "forecast_dayprior"]
if not INCLUDE_RADIATION_DAY_BEFORE: # HEY TEAM I ADDED THIS LIUNE HERE
    DROP_COLS.append("radiation_day_before")
FEATURES = [c for c in train.columns if c not in DROP_COLS]

len(FEATURES)

48

In [4]:
# iterating through different hyperparameters to tune

param_grid = [
    {"n_estimators": n, "max_depth": d}
    for n in [100, 300]
    for d in [None, 10, 20]
]

best_rmse = None
best_model = None
best_params = None

for params in param_grid:
    rf = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    rf.fit(train[FEATURES], train[TARGET])

    val_pred = rf.predict(validation[FEATURES])
    rmse = np.sqrt(mean_squared_error(validation[TARGET], val_pred))
    print(params, f"val rmse: {rmse:.2f}")

    if best_rmse is None or rmse < best_rmse:
        best_rmse = rmse
        best_model = rf
        best_params = params

print()
print(f"best params: {best_params}, val rmse: {best_rmse:.2f}")
model = best_model

{'n_estimators': 100, 'max_depth': None} val rmse: 442.95
{'n_estimators': 100, 'max_depth': 10} val rmse: 456.64
{'n_estimators': 100, 'max_depth': 20} val rmse: 441.32
{'n_estimators': 300, 'max_depth': None} val rmse: 441.99
{'n_estimators': 300, 'max_depth': 10} val rmse: 456.41
{'n_estimators': 300, 'max_depth': 20} val rmse: 440.68

best params: {'n_estimators': 300, 'max_depth': 20}, val rmse: 440.68


In [5]:
# only predicting on test now that the model/params are picked using validation above

predictions = model.predict(test[FEATURES])

In [6]:
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2 = r2_score(y_true, y_pred)
    metrics = {"model": model_name, "rmse": rmse, "mae": mae, "mape_pct": mape, "r2": r2}
    print(metrics)
    return metrics


metrics = evaluate(test[TARGET], predictions, MODEL_NAME)

{'model': 'random_forest', 'rmse': np.float64(493.69677832091224), 'mae': 322.4514120820744, 'mape_pct': 3.927903268154292, 'r2': 0.8439689707979023}


In [7]:
# saving prediction performance

pd.Series(predictions, index=test["DATETIME"], name=MODEL_NAME).to_csv(RESULTS_DIR / f"{MODEL_NAME}_predictions.csv")

comparison_path = RESULTS_DIR / "model_comparison.csv"
this_run = pd.DataFrame([metrics])

if comparison_path.exists():
    existing = pd.read_csv(comparison_path)
    existing = existing[existing["model"] != MODEL_NAME]
    this_run = pd.concat([existing, this_run], ignore_index=True)

this_run.to_csv(comparison_path, index=False)
this_run

,model,rmse,mae,mape_pct,r2,comments
0,xgboost,470.544565,311.176400,3.791660,0.858260,NaN
1,lightgbm,463.995139,307.403004,3.744587,0.862178,NaN
2,baseline_linear_regression,535.989120,380.215288,4.743792,0.816091,\r\nLinear Regression on the 48 shared feature...
3,prophet,469.383322,345.777006,4.289860,0.843310,NaN
4,random_forest_no_radiation_day_before,502.050043,326.322267,3.962717,0.838644,NaN
5,aemo_forecast_closest,64.292411,47.711611,0.595688,0.997354,NaN
6,aemo_forecast_12hr_prior,212.913694,155.945260,1.921867,0.970980,NaN
7,aemo_forecast_dayprior,222.460754,161.830989,1.988398,0.968319,NaN
8,random_forest,493.696778,322.451412,3.927903,0.843969,NaN


In [8]:
# out of curiosoity lookinag at the feature importances 

importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
importances

demand_day_before         0.810455
day_Monday                0.044651
day_Saturday              0.030768
radiation_day_before      0.020477
temperature_day_before    0.019939
is_public_holiday         0.010284
hour_7                    0.005886
day_Tuesday               0.005536
day_Sunday                0.004830
hour_0                    0.004485
hour_6                    0.004126
hour_8                    0.003138
month_1                   0.002847
month_12                  0.002776
hour_1                    0.002477
is_school_term            0.002393
hour_21                   0.001563
hour_23                   0.001397
hour_20                   0.001392
hour_22                   0.001362
month_2                   0.001351
month_11                  0.001344
hour_5                    0.001225
hour_19                   0.001201
day_Friday                0.001033
hour_18                   0.001025
month_10                  0.000913
hour_2                    0.000911
hour_9              

In [9]:
# checking the aemo performance 

aemo_raw = pd.read_csv(DATA_DIR / "nsw_features_added.csv", parse_dates=["DATETIME"])
aemo_test = aemo_raw[aemo_raw["DATETIME"].dt.year == 2019]

aemo_forecasts = {
    "aemo_forecast_closest": "forecast_closest",
    "aemo_forecast_12hr_prior": "forecast_12hr_prior",
    "aemo_forecast_dayprior": "forecast_dayprior",
}

aemo_metrics = [evaluate(aemo_test["TOTALDEMAND"], aemo_test[col], name) for name, col in aemo_forecasts.items()]

{'model': 'aemo_forecast_closest', 'rmse': np.float64(64.29241067137875), 'mae': 47.71161130136987, 'mape_pct': 0.5956884401924603, 'r2': 0.9973538783442838}
{'model': 'aemo_forecast_12hr_prior', 'rmse': np.float64(212.91369389390076), 'mae': 155.9452602739726, 'mape_pct': 1.9218666283899966, 'r2': 0.9709799887677941}
{'model': 'aemo_forecast_dayprior', 'rmse': np.float64(222.4607535563383), 'mae': 161.83098858447488, 'mape_pct': 1.9883980439297608, 'r2': 0.9683191230406595}


In [10]:
# ading aemo into the csv

existing = pd.read_csv(comparison_path)
existing = existing[~existing["model"].isin(aemo_forecasts.keys())]
this_run = pd.concat([existing, pd.DataFrame(aemo_metrics)], ignore_index=True)
this_run.to_csv(comparison_path, index=False)
this_run

,model,rmse,mae,mape_pct,r2,comments
0,xgboost,470.544565,311.176400,3.791660,0.858260,NaN
1,lightgbm,463.995139,307.403004,3.744587,0.862178,NaN
2,baseline_linear_regression,535.989120,380.215288,4.743792,0.816091,\r\nLinear Regression on the 48 shared feature...
3,prophet,469.383322,345.777006,4.289860,0.843310,NaN
4,random_forest_no_radiation_day_before,502.050043,326.322267,3.962717,0.838644,NaN
5,random_forest,493.696778,322.451412,3.927903,0.843969,NaN
6,aemo_forecast_closest,64.292411,47.711611,0.595688,0.997354,NaN
7,aemo_forecast_12hr_prior,212.913694,155.945260,1.921867,0.970980,NaN
8,aemo_forecast_dayprior,222.460754,161.830989,1.988398,0.968319,NaN
